# Homework: Agentic RAG

In this homework, we build a RAG system from scratch and then make it agentic - the same path as the module.

Instead of the course FAQ, our knowledge base is the course lessons themselves.

The course repository is organized by module. Each module is a top-level folder with a lessons/ subfolder of numbered markdown pages:

```
01-agentic-rag/
└── lessons/
    ├── 01-intro.md
    ├── 02-environment.md
    ├── ...
    └── 16-other-frameworks.md
```

There are seven modules:

- 01-agentic-rag
- 02-vector-search
- 03-orchestration
- 04-evaluation
- 05-monitoring
- 06-best-practices
- 07-project-example

Each lesson page is a single markdown file. These pages are exactly what you read as you go through the course.

We'll fetch this data from GitHub and use it as the knowledge base for our RAG system.

`It's possible your answers won't match exactly. If so, select the closest one.`

### Setup

Prepare your environment the same way as in the module's Environment lesson.

This homework needs one extra library: gitsource, which downloads files from a GitHub repository.

Install it:

`uv add gitsource`

For the LLM, we recommend OpenAI with gpt-5.4-mini, but you can use any model and provider you like - just adapt the client and the usage fields accordingly.

### Preparation

First, we will pull the lesson pages straight from the course repository. We will use the commit 8c1834d to make sure everyone works with the exact same data.

We will use `gitsource` for that:

```python

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

```

`GithubRepositoryDataReader` downloads the entire repository and goes over all the files in it. Because we specify `allowed_extensions={"md"}`, it only checks the markdown files.

We also pass a `filename_filter` so we don't grab every markdown file in the repo, like the top-level README. The lesson pages all live under a module's `lessons/` folder, so filtering on `/lessons/` keeps just those.

Each file has a `parse()` method that returns a dictionary with its `filename` and `content`:

```python 

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

```

### Q1. How many lesson pages

How many lesson pages are in the dataset?

- 24
- 72
- 240
- 720

72

In [1]:
from gitsource import GithubRepositoryDataReader
from minsearch import Index

In [2]:

reader = GithubRepositoryDataReader(
repo_owner="DataTalksClub",
repo_name="llm-zoomcamp",
commit_id="8c1834d",
allowed_extensions={"md"},
filename_filter=lambda path: "/lessons/" in path,
)
files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)
    
   

In [3]:
print(len(documents))

72


# Calling LLM API

In [4]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from openai import OpenAI
client = OpenAI()

### Q2. Indexing and searching

Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:

How does the agentic loop keep calling the model until it stops?

What's the filename of the first result?

- 01-agentic-rag/lessons/03-rag.md
- 01-agentic-rag/lessons/14-agentic-loop.md
- 04-evaluation/lessons/13-llm-as-judge.md
- 06-best-practices/lessons/02-hybrid-search.md


'filename': '01-agentic-rag/lessons/14-agentic-loop.md'

In [6]:
index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

index.fit(documents)

In [7]:
def search(question,):
    boost_dict = {'content': 2.0, 'filename': 0.5}

    return index.search(
        question,
        boost_dict=boost_dict,
        num_results=5
    )

In [8]:
question = "How does the agentic loop keep calling the model until it stops?"

search_result = search(question)

search_result

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

### Q3. RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper script we prepared during the lessons:

`wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py`

`RAGBase` was written for the FAQ schema (`section`/`question`/`answer`), while our documents have `filename` and `content`.

Two solutions are possible:

- Implement the `RAG flow` yourself
- Take `RAGBase` and change the parts related to the FAQ schema - `search` (to use our index) and `build_context`

Build a RAG over the index from Q2 and answer the query:

`How does the agentic loop keep calling the model until it stops?`

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

- 700
- 7000
- 70000
- 700000

7135

We count input tokens instead of price because the cost depends on the model and provider you use, but the size of the prompt we send is the same for everyone.

Most LLM APIs report token usage on the response object (e.g. `response.usage.input_tokens` / `prompt_tokens`). We'll read the input tokens from there.

You will need to modify the code for the rag helper to expose the usage.

In the RAG Helper class, `llm` returns only the text. Modify it to return the whole response, and change `rag` to return both the answer and usage (as a tuple or create a small dataclass for that).

In [14]:
from rag_helper import RAGBase

assistant = RAGBase(index, client)

In [10]:
search_results = assistant.search(question)
context = assistant.build_context(search_results)
prompt = assistant.build_prompt(question, search_results)
response = assistant.llm(prompt)
rag = assistant.rag(question)

In [11]:
print(response.output_text)

The agentic loop keeps calling the model inside a `while True` loop.

After each model response, it checks whether the response contains any `function_call` items:

- If there is a function call, it runs the tool, appends the tool result to the message history, and loops again.
- If there are no function calls, it breaks out of the loop and stops.

So the stop condition is פשוט: **no function calls this turn**.


In [12]:
print(response.usage)

ResponseUsage(input_tokens=7135, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=97, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7232)


### Q4. Chunking

The lesson pages are long - some are thousands of characters. Long documents make retrieval less precise: a match deep inside a page still pulls in the whole page. A common fix is chunking: split each page into smaller, overlapping pieces and index those instead.

gitsource has a helper for this: `chunk_documents`. It uses a sliding window - a window of `size` characters slides across the text in steps of step characters, and each window position becomes one chunk:

```python

from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

```

With `size=2000` and `step=1000` (you can see the implementation here):

- Each chunk is a window of `size` characters of the page.
- The window moves forward by `step` characters between chunks. Since `step` is smaller than `size`, consecutive chunks overlap by `size - step` (1000) characters, so a passage split across a boundary still appears whole in one of the chunks.
- Every chunk keeps the original fields (`filename`) and adds `start` (the offset in the page) and `content` (the chunk text).

How many chunks do you get?

- 70
- 295
- 1100
- 4500

295

In [15]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [17]:
print(len(chunks))

295


In [27]:
chunks[100]

{'start': 0,
 'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWORD

### Q5. RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the LLM. Let's measure that.

Index the chunks from Q4 (same as before: content as a text field, filename as a keyword field), point your RAG at the chunk index, and answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

- about the same
- 3× fewer
- 10× fewer
- 30× fewer

2318 : 3x fewer

In [23]:
# using the chunks to create a new index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

index.fit(chunks)

In [20]:
assistant = RAGBase(index, client)
search_results = assistant.search(question)
context = assistant.build_context(search_results)
prompt = assistant.build_prompt(question, search_results)
response = assistant.llm(prompt)
rag = assistant.rag(question)

In [21]:
print(response.output_text)

It keeps looping with a `while True` and uses a `has_function_calls` flag.

- Each turn, it calls the model.
- If the model returns any `function_call` items, the code runs them, appends the outputs, and sets `has_function_calls = True`.
- If the model returns only a final `message` and no function calls, `has_function_calls` stays `False`, and the loop `break`s.

So the stop condition is: **no function calls in the model response**.


In [22]:
print(response.usage)

ResponseUsage(input_tokens=2318, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=111, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2429)


### Q6. Turning it into an agent

So far search runs once, with the exact query. Let's make it agentic: give the LLM a `search` tool and let it decide when (and what) to search. We suggest toyaikit, the small agent library from the module, but you can use anything you like - the OpenAI Agents SDK, PydanticAI, LangChain, or a hand-written loop.

If you go with toyaikit:

`uv add toyaikit`

Create a `search` function that uses the chunk index. Give it a type hint and a docstring - most frameworks read them to build the tool schema for you.

Build an agent with your `search` tool and run it (with toyaikit, the same way as in the ToyAIKit lesson). Use these instructions for the agent (they nudge it to search a few times):

```

You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.

```
Ask it:

`How does the agentic loop work, and how is it different from plain RAG?`

The agent decides on its own when to search and when to answer. Count how many times it called the `search` tool.

How many times did the agent call `search`?

```

Note: the agent decides this itself, so it varies a little between runs - pick the closest option. We measured this with OpenAI `gpt-5.4-mini`; with a different model or provider the number may differ, so keep that in mind.

```

- 0
- 4
- 10
- 20

6

In [44]:
from agents import Agent, Runner, function_tool, ToolCallItem

In [41]:
@function_tool
def search_chunks(query: str) -> list:
    """ Search the course lessons for content relevant to the question.

    Args:
        query: Search query text to look up in the course lessons.
    """

    return index.search(query, num_results=1)

In [42]:
print(type(search_chunks))

agent = Agent(
    name="Course Assistant",
    instructions="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.",
    model="gpt-5.4-mini",
    tools=[search_chunks],
)

<class 'agents.tool.FunctionTool'>


In [43]:
result = await Runner.run(
    agent,
    "How does the agentic loop work, and how is it different from plain RAG?"
)

print(result.final_output)

The agentic loop is a repeated **think → act → observe → think** cycle:

1. the model gets the user goal
2. it decides whether to answer directly or use a tool
3. it performs the tool action
4. it reads the result
5. it repeats until it can respond

So instead of one retrieval + one generation step, it can make **multiple model calls** and **multiple tool calls** as needed, keeping the conversation/history in `previous_messages` and returning the full `all_messages`.

Plain RAG is simpler:

- **retrieve relevant chunks**
- **stuff them into the prompt**
- **generate one answer**

So the main difference is:

- **RAG:** one-shot retrieval grounding
- **Agentic loop:** iterative, tool-using workflow that can retrieve, reason, act again, and keep state

In short: RAG answers from context; agentic loops can **decide what to do next** and keep going.


In [47]:
n_calls = sum(1 for item in result.new_items if isinstance(item, ToolCallItem))
print(f"Agent called search_chunks {n_calls} times")


Agent called search_chunks 6 times
